<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Galería · Gráficos para informes
**Excel Corporativo · 12 Claves** — Los 12 gráficos más usados, con su ficha de decisión · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/graficos_para_informes.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#graficos)

**Al terminar podrás:**
1. Elegir el gráfico según lo que quieres mostrar.
2. Escribir títulos que dicen el hallazgo.
3. Reconocer las decisiones de diseño que hacen legible un gráfico.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## Cómo elegir
| Si quieres mostrar… | Usa |
|---|---|
| **Comparación** entre categorías | Barras o columnas agrupadas |
| **Evolución** en el tiempo | Líneas |
| **Composición** de un total | Apiladas al 100 %, anillo (máximo 4 partes) o cascada |
| **Distribución** de valores | Histograma, mapa de calor o Pareto |
| **Relación** entre dos medidas | Dispersión |
| **Cumplimiento** de una meta | Indicadores (KPI) o real contra meta |

Elige una pregunta en la lista y ejecuta la celda. Debajo del gráfico aparece su ficha de decisión.

In [ ]:
# @title Elige la pregunta y mira el gráfico
pregunta = "Barras: ¿qué sede vendió más?"  # @param ["Barras: ¿qué sede vendió más?", "Columnas agrupadas: ¿cómo cambió cada sede?", "Líneas: ¿cómo se comportó la venta mes a mes?", "Apiladas 100 %: ¿el e-commerce ganó participación?", "Anillo: ¿qué segmento pesa más?", "Histograma: ¿cuánto vale una factura típica?", "Dispersión: ¿vender más da más margen?", "Mapa de calor: ¿dónde y cuándo se concentra la venta?", "Pareto: ¿qué tan concentrados están los clientes?", "Indicadores (KPI): ¿cómo cerró el año?", "Real contra meta: ¿quién cumplió?", "Cascada: ¿qué explica el crecimiento?"]
v25, v24 = validas[validas.anio == 2025], validas[validas.anio == 2024]
fichas = {}
def F(usa, evita, excel, decisiones): return f"**Úsalo cuando:** {usa}  \n**Evítalo cuando:** {evita}  \n**En Excel 365:** {excel}  \n**Decisiones:** {decisiones}"
k = pregunta.split(":")[0]
if k == "Barras":
    s = v25.groupby("ciudad").subtotal.sum().sort_values() / 1e6
    fig = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h", marker_color=[GRIS] * 5 + [AZUL], text=[f"$ {v:,.0f} M".replace(",", ".") for v in s.values], textposition="outside", cliponaxis=False))
    fig.update_layout(title=f"{s.index[-1]} concentra el {porc(s.iloc[-1] / s.sum())} de la venta 2025", xaxis_title=MM); dar_aire(fig, s.max())
    txt = F("comparas categorías, sobre todo si son muchas o tienen nombres largos.", "el eje es el tiempo.", "Insertar › Barra agrupada, con los datos ordenados.", "ordenar de mayor a menor, un color y un resaltado, eje desde cero.")
elif k == "Columnas agrupadas":
    p = validas.pivot_table(index="ciudad", columns="anio", values="subtotal", aggfunc="sum").loc[ORDEN_CIUDADES] / 1e6
    fig = go.Figure([go.Bar(name="2024", x=p.index, y=p[2024], marker_color=GRIS), go.Bar(name="2025", x=p.index, y=p[2025], marker_color=AZUL)])
    fig.update_layout(barmode="group", title=f"La venta total creció {porc(p[2025].sum() / p[2024].sum() - 1)} de 2024 a 2025", yaxis_title=MM)
    txt = F("comparas dos o tres series dentro de cada categoría.", "hay más de tres series.", "Insertar › Columna agrupada.", "año anterior en gris, actual en color, máximo tres series.")
elif k == "Líneas":
    m = validas.groupby(["anio", "mes"]).subtotal.sum().unstack(0) / 1e6
    fig = go.Figure([go.Scatter(x=MESES, y=m[a], name=str(a), mode="lines+markers", line=dict(color=c)) for a, c in [(2024, GRIS), (2025, AZUL)]])
    fig.update_layout(title=f"El mes más fuerte de 2025 fue {MESES[int(m[2025].idxmax()) - 1].lower()}", yaxis_title=MM)
    txt = F("el eje horizontal es el tiempo.", "las categorías no tienen orden.", "Insertar › Línea con marcadores.", "máximo cuatro líneas, etiquetas al final.")
elif k == "Apiladas 100 %":
    p = pd.crosstab(validas.anio, validas.canal, values=validas.subtotal, aggfunc="sum", normalize="index")[["Presencial", "Televentas", "E-commerce"]]
    fig = go.Figure([go.Bar(name=c, y=[str(a) for a in p.index], x=p[c], orientation="h", marker_color=col, text=[f"{v:.0%}" for v in p[c]], marker_line_color="white", marker_line_width=2) for c, col in zip(p.columns, [AZUL, TEAL, MAGENTA])])
    fig.update_layout(barmode="stack", xaxis_tickformat=".0%", title=f"El e-commerce pasó de {porc(p.iloc[0, 2])} a {porc(p.iloc[1, 2])}", yaxis=dict(autorange="reversed"), height=320)
    txt = F("comparas cómo se reparte un total entre pocas partes.", "importan los valores absolutos.", "Insertar › Barra 100 % apilada.", "pocas partes, mismo orden, porcentajes dentro.")
elif k == "Anillo":
    s = v25.groupby("segmento").subtotal.sum().sort_values(ascending=False)
    fig = go.Figure(go.Pie(labels=s.index, values=s.values, hole=0.55, sort=False, marker_colors=[AZUL, TEAL, MAGENTA, GRIS], textinfo="label+percent"))
    fig.update_layout(title=f"{s.index[0]} aporta el {porc(s.iloc[0] / s.sum())} de la venta 2025", showlegend=False)
    txt = F("una sola serie, máximo cuatro partes y una parte domina.", "hay más de cuatro partes o son parecidas.", "Insertar › Anillo. Nunca en 3D.", "ordenar desde las 12, escribir porcentajes.")
elif k == "Histograma":
    fac = v25.groupby("id_factura").subtotal.sum() / 1e6; tope = fac.quantile(0.95)
    fig = px.histogram(fac.clip(upper=tope), nbins=20, color_discrete_sequence=[TEAL])
    fig.update_layout(title=f"La mitad de las facturas vale menos de $ {fac.median():,.1f} M".replace(".", ","), xaxis_title=f"Valor de la factura (millones; la última barra agrupa las mayores a {tope:,.0f} M)", yaxis_title="Facturas", showlegend=False)
    txt = F("quieres ver qué valores son típicos y cuáles extremos.", "tienes pocos datos o categorías.", "Insertar › Gráfico estadístico › Histograma.", "ancho de intervalo, agrupar extremos, reportar la mediana.")
elif k == "Dispersión":
    g = validas.groupby("nombre_vendedor").agg(venta=("subtotal", "sum"), margen=("margen_bruto", "sum")); g["margen_pct"] = g.margen / g.venta
    r = np.corrcoef(g.venta, g.margen_pct)[0, 1]
    fig = px.scatter(g.reset_index(), x="venta", y="margen_pct", hover_name="nombre_vendedor", color_discrete_sequence=[AZUL])
    fig.update_traces(marker_size=12); fig.update_layout(yaxis_tickformat=".0%", xaxis_title="Venta 2024–2025 (pesos)", yaxis_title="Margen bruto %",
        title="Los que más venden tienden a dejar menos margen" if r < -0.3 else "Vender más no se asocia con más margen")
    txt = F("quieres ver si dos medidas se mueven juntas.", "una variable es categoría o fecha.", "Insertar › Dispersión (X, Y).", "variable explicativa en X, rotular solo lo importante; relación no es causa.")
elif k == "Mapa de calor":
    p = v25.pivot_table(index="ciudad", columns="mes", values="subtotal", aggfunc="sum").loc[ORDEN_CIUDADES] / 1e6
    fig = px.imshow(p, x=MESES, text_auto=".0f", aspect="auto", color_continuous_scale=["#EEF2FF", AZUL, "#00006A"]); fig.update_layout(title="Venta 2025 por sede y mes (millones)")
    txt = F("cruzas dos categorías y buscas patrones.", "necesitas comparar cifras exactas.", "Formato condicional › Escalas de color.", "un solo color, filas ordenadas, valores escritos.")
elif k == "Pareto":
    c = v25.groupby("razon_social").subtotal.sum().sort_values(ascending=False); n = len(c)
    dec = pd.Series([c.iloc[int(n * d / 10):int(n * (d + 1) / 10)].sum() / c.sum() for d in range(10)], index=[f"{(d + 1) * 10} %" for d in range(10)])
    fig = go.Figure([go.Bar(x=dec.index, y=dec.values, name="Aporte", marker_color=AZUL), go.Scatter(x=dec.index, y=dec.cumsum(), name="Acumulado", mode="lines+markers", line=dict(color=MAGENTA))])
    fig.add_hline(y=0.8, line_dash="dot", line_color=MAGENTA)
    fig.update_layout(yaxis_tickformat=".0%", title=f"El 20 % de los clientes explica el {porc(dec.iloc[:2].sum())} de la venta", xaxis_title="Clientes agrupados del que más compra al que menos")
    txt = F("quieres saber si pocos explican la mayoría.", "todos aportan parecido.", "Insertar › Gráfico estadístico › Pareto.", "ordenar, marcar el 80 %, agrupar si son muchos.")
elif k == "Indicadores (KPI)":
    fig = go.Figure([go.Indicator(mode="number+delta", value=v25.subtotal.sum() / 1e6, delta=dict(reference=v24.subtotal.sum() / 1e6, relative=True, valueformat=".1%"), number=dict(prefix="$ ", suffix=" M", valueformat=",.0f"), title="Venta neta 2025", domain=dict(x=[0, 0.3])),
                     go.Indicator(mode="number", value=v25.margen_bruto.sum() / v25.subtotal.sum() * 100, number=dict(suffix=" %", valueformat=".1f"), title="Margen bruto", domain=dict(x=[0.35, 0.65])),
                     go.Indicator(mode="number", value=v25.id_factura.nunique(), number=dict(valueformat=","), title="Facturas", domain=dict(x=[0.7, 1]))])
    fig.update_layout(title="Lo primero que ve el comité", height=280)
    txt = F("el mensaje es una cifra y su comparación.", "pones más de cinco tarjetas.", "Celda grande con IMPORTARDATOSDINAMICOS.", "toda cifra con su comparación; color solo en la comparación.")
elif k == "Real contra meta":
    real = v25.groupby("ciudad").subtotal.sum(); meta = metas[metas.anio == 2025].merge(sedes, on="id_sede").groupby("ciudad").meta_subtotal.sum()
    cum = (real / meta).sort_values()
    fig = go.Figure(go.Bar(x=cum.values, y=cum.index, orientation="h", marker_color=[TEAL if v >= 1 else MAGENTA for v in cum.values], text=[("✓ " if v >= 1 else "✗ ") + porc(v) for v in cum.values], textposition="outside"))
    fig.add_vline(x=1, line_dash="dot"); fig.update_layout(xaxis_tickformat=".0%", xaxis_range=[0.8, cum.max() + 0.08], title=f"{(cum >= 1).sum()} de 6 sedes cumplieron la meta 2025")
    txt = F("comparas un resultado con su objetivo.", "no hay meta definida.", "Barra agrupada con superposición 100 %.", "ordenar por cumplimiento, % escrito, ✓ y ✗ con el color.")
else:
    p = validas.pivot_table(index="ciudad", columns="anio", values="subtotal", aggfunc="sum"); d = (p[2025] - p[2024]).sort_values(ascending=False)
    fig = go.Figure(go.Waterfall(x=["2024", *d.index, "2025"], y=[p[2024].sum() / 1e6, *(d.values / 1e6), 0], measure=["absolute", *["relative"] * len(d), "total"],
                                 increasing=dict(marker_color=BIEN), decreasing=dict(marker_color=MAL), totals=dict(marker_color=GRIS)))
    fig.update_layout(title=f"{d.index[0]} aportó la mayor parte del crecimiento", yaxis_title=MM)
    txt = F("explicas cómo se pasa de una cifra a otra.", "los pasos no suman el total.", "Insertar › Cascada; marca los totales con «Establecer como total».", "totales en gris, aumentos en verde, disminuciones en rojo.")
fig.update_layout(legend_title="", height=fig.layout.height or 420)
fig.show()
display(Markdown(txt))

## Seis reglas para cualquier gráfico
1. **Empieza por la pregunta**, no por el tipo de gráfico.
2. **El título es el hallazgo**: una frase con la cifra que importa.
3. **Ordena** de mayor a menor, salvo que el orden sea el tiempo.
4. **Un color y un resaltado**: el color señala, no decora.
5. **Escribe los valores clave** si el lector necesita la cifra exacta.
6. **Fuente y fecha** al pie.

**Evita:** gráficos en 3D, dos ejes con escalas distintas, tortas con muchas partes y barras que no empiezan en cero.

## Retos: elige el gráfico

In [ ]:
# @title Reto 1 · Quieres mostrar cómo cambió la venta mes a mes durante dos años
respuesta = "Elige una opción"  # @param ["Elige una opción", "Barras", "Columnas agrupadas", "Líneas", "Apiladas 100 %", "Anillo", "Histograma"]
revisar(respuesta, "Líneas", """El eje horizontal es el tiempo.""")

In [ ]:
# @title Reto 2 · Quieres mostrar que pocos clientes explican la mayor parte de la venta
respuesta = "Elige una opción"  # @param ["Elige una opción", "Anillo", "Pareto", "Dispersión", "Líneas"]
revisar(respuesta, "Pareto", """El Pareto muestra el aporte acumulado y marca el 80 %.""")

In [ ]:
# @title Reto 3 · Quieres mostrar qué sedes llegaron a su meta
respuesta = "Elige una opción"  # @param ["Elige una opción", "Mapa de calor", "Histograma", "Real contra meta", "Cascada"]
revisar(respuesta, "Real contra meta", """Compara el resultado con el objetivo, sede por sede.""")

## Resumen
* Pregunta → intención → gráfico → título con el hallazgo.

**Siguiente:** [Laboratorio Excel + Colab](https://omarli-360.github.io/excel-corporativo-12-claves/#laboratorio)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>